# **Text Mining - Assignment 1**

**Character**: *Rachel*

**Group 1** - Elisabeth Filippini, Emerentia Arjani Nurman, Dhruv Gupta, Horia Cătălin

In [132]:
import pandas as pd

### **Text Preprocessing**

First, we begin by loading the dataset into the notebook environment:

In [133]:
df = pd.read_csv('Friends.csv')
df.head()

,Text,Speaker,Episode,Season,Show
0,Originally written by Marta Kauffman and David...,NaN,Episode-01-The One Where Monica Gets a New Roo...,Season-01,Friends
1,Transcribed by guineapig.,NaN,Episode-01-The One Where Monica Gets a New Roo...,Season-01,Friends
2,CENTRAL PERK. (ALL PRESENT EXCEPT RACHEL AND ...,SCENE 1,Episode-01-The One Where Monica Gets a New Roo...,Season-01,Friends
3,There's nothing to tell! He's just some guy I...,MONICA,Episode-01-The One Where Monica Gets a New Roo...,Season-01,Friends
4,"C'mon, you're going out with the guy! There's...",JOEY,Episode-01-The One Where Monica Gets a New Roo...,Season-01,Friends


Keeping only the `text` column for the preprocessing steps:

In [134]:
original_text = pd.DataFrame(df['Text'])
original_text

,Text
0,Originally written by Marta Kauffman and David...
1,Transcribed by guineapig.
2,CENTRAL PERK. (ALL PRESENT EXCEPT RACHEL AND ...
3,There's nothing to tell! He's just some guy I...
4,"C'mon, you're going out with the guy! There's..."
...,...
69969,Then I'm happy too. (They're still hugging - ...
69970,COMMERCIAL BREAK
69971,Estelle's memorial service. Joey is giving a ...
69972,Thank you all for coming. We're here today to...


**a. Removing punctuations such as  ’ !”#$%&()*+,-.:;?@[]\ˆ ‘{|}∼’**

In [135]:
chars_to_remove = r'''!"#$%&()*+,-./:;<=>?@[\]^_`{|}~—–'''
table = str.maketrans("", "", chars_to_remove)
text = original_text.copy()

text['Text'] = text['Text'].apply(lambda x: x.translate(table))
text

,Text
0,Originally written by Marta Kauffman and David...
1,Transcribed by guineapig
2,CENTRAL PERK ALL PRESENT EXCEPT RACHEL AND ROSS
3,There's nothing to tell He's just some guy I ...
4,C'mon you're going out with the guy There's g...
...,...
69969,Then I'm happy too They're still hugging fad...
69970,COMMERCIAL BREAK
69971,Estelle's memorial service Joey is giving a s...
69972,Thank you all for coming We're here today to ...


**b. Lower case the text**

In [136]:
text['Text'] = text['Text'].str.lower()
text

,Text
0,originally written by marta kauffman and david...
1,transcribed by guineapig
2,central perk all present except rachel and ross
3,there's nothing to tell he's just some guy i ...
4,c'mon you're going out with the guy there's g...
...,...
69969,then i'm happy too they're still hugging fad...
69970,commercial break
69971,estelle's memorial service joey is giving a s...
69972,thank you all for coming we're here today to ...


**c. Perform sentence and word tokenization on every script**

For this, we will employ Tokenization via **Regular Expressions (`RegEx`)** to perform both word and sentence tokenization.

First, we start with **word** tokenization:

In [137]:
import re

text['Tokens'] = text['Text'].str.findall(r"[\w']+")
text['Tokens']

0        [originally, written, by, marta, kauffman, and...
1                             [transcribed, by, guineapig]
2        [central, perk, all, present, except, rachel, ...
3        [there's, nothing, to, tell, he's, just, some,...
4        [c'mon, you're, going, out, with, the, guy, th...
                               ...                        
69969    [then, i'm, happy, too, they're, still, huggin...
69970                                  [commercial, break]
69971    [estelle's, memorial, service, joey, is, givin...
69972    [thank, you, all, for, coming, we're, here, to...
69973                                           [the, end]
Name: Tokens, Length: 69974, dtype: object

Now **sentence** tokenization:

However, to perform sentence tokenization, we must go back to the original text with punctuations:
- We want to split the punctuated text at . ! ? followed by a whitespace, but not after Mr./Mrs./Dr./Ms.

In [138]:
sentence_end = r'(?<!\bmr\.)(?<!\bmrs\.)(?<!\bdr\.)(?<!\bms\.)(?<=[.!?])\s+'
text['Sentences'] = original_text['Text'].str.lower().str.split(sentence_end, regex=True)

# Remove punctuation from each sentence and drop empty ones
text['Sentences'] = text['Sentences'].apply(
    lambda sents: [' '.join(s.translate(table).split()) for s in sents])
text['Sentences'] = text['Sentences'].apply(lambda sents: [s for s in sents if s])
text

,Text,Tokens,Sentences
0,originally written by marta kauffman and david...,"[originally, written, by, marta, kauffman, and...",[originally written by marta kauffman and davi...
1,transcribed by guineapig,"[transcribed, by, guineapig]",[transcribed by guineapig]
2,central perk all present except rachel and ross,"[central, perk, all, present, except, rachel, ...","[central perk, all present except rachel and r..."
3,there's nothing to tell he's just some guy i ...,"[there's, nothing, to, tell, he's, just, some,...","[there's nothing to tell, he's just some guy i..."
4,c'mon you're going out with the guy there's g...,"[c'mon, you're, going, out, with, the, guy, th...","[c'mon you're going out with the guy, there's ..."
...,...,...,...
69969,then i'm happy too they're still hugging fad...,"[then, i'm, happy, too, they're, still, huggin...","[then i'm happy too, they're still hugging fad..."
69970,commercial break,"[commercial, break]",[commercial break]
69971,estelle's memorial service joey is giving a s...,"[estelle's, memorial, service, joey, is, givin...","[estelle's memorial service, joey is giving a ..."
69972,thank you all for coming we're here today to ...,"[thank, you, all, for, coming, we're, here, to...","[thank you all for coming, we're here today to..."


**d. Remove stopwords using the `NLTK` library**

**Stopwords** are very common words that carry little meaning on their own. They appear in almost every document, so they say little about what a particular text is about.

In [139]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Horia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Removing stopwords from words & sentences:

In [140]:
stop_words = set(stopwords.words('english'))

text['Tokens_no_stop'] = text['Tokens'].apply(
    lambda tokens: [w for w in tokens if w not in stop_words])
text[['Tokens', 'Tokens_no_stop']]

,Tokens,Tokens_no_stop
0,"[originally, written, by, marta, kauffman, and...","[originally, written, marta, kauffman, david, ..."
1,"[transcribed, by, guineapig]","[transcribed, guineapig]"
2,"[central, perk, all, present, except, rachel, ...","[central, perk, present, except, rachel, ross]"
3,"[there's, nothing, to, tell, he's, just, some,...","[there's, nothing, tell, guy, work]"
4,"[c'mon, you're, going, out, with, the, guy, th...","[c'mon, going, guy, there's, gotta, something,..."
...,...,...
69969,"[then, i'm, happy, too, they're, still, huggin...","[happy, still, hugging, fade]"
69970,"[commercial, break]","[commercial, break]"
69971,"[estelle's, memorial, service, joey, is, givin...","[estelle's, memorial, service, joey, giving, s..."
69972,"[thank, you, all, for, coming, we're, here, to...","[thank, coming, today, pay, respects, wonderfu..."


In [141]:
text['Sentences_no_stop'] = text['Sentences'].apply(
    lambda sents: [' '.join(w for w in s.split() if w not in stop_words) for s in sents]
)
text[['Sentences', 'Sentences_no_stop']]

,Sentences,Sentences_no_stop
0,[originally written by marta kauffman and davi...,[originally written marta kauffman david crane]
1,[transcribed by guineapig],[transcribed guineapig]
2,"[central perk, all present except rachel and r...","[central perk, present except rachel ross]"
3,"[there's nothing to tell, he's just some guy i...","[there's nothing tell, guy work]"
4,"[c'mon you're going out with the guy, there's ...","[c'mon going guy, there's gotta something wrong]"
...,...,...
69969,"[then i'm happy too, they're still hugging fad...","[happy, still hugging fade]"
69970,[commercial break],[commercial break]
69971,"[estelle's memorial service, joey is giving a ...","[estelle's memorial service, joey giving speec..."
69972,"[thank you all for coming, we're here today to...","[thank coming, today pay respects wonderful ag..."


**d. Use the Porter stemmer to perform stemming on the used words in the scripts**

**Stemming** reduces a word to its root by chopping off suffixes with fixed rules, so different forms of a word map to the same token.

In [142]:
from nltk.stem import PorterStemmer

# Instantiatinng the Porter Stemmer Class
ps = PorterStemmer()

text['Tokens_stem'] = text['Tokens'].apply(lambda tokens: [ps.stem(w) for w in tokens])
text['Tokens_stem']

0        [origin, written, by, marta, kauffman, and, da...
1                               [transcrib, by, guineapig]
2        [central, perk, all, present, except, rachel, ...
3        [there', noth, to, tell, he', just, some, guy,...
4        [c'mon, you'r, go, out, with, the, guy, there'...
                               ...                        
69969    [then, i'm, happi, too, they'r, still, hug, fa...
69970                                    [commerci, break]
69971    [estelle', memori, servic, joey, is, give, a, ...
69972    [thank, you, all, for, come, we'r, here, today...
69973                                           [the, end]
Name: Tokens_stem, Length: 69974, dtype: object

**e. Perform POS tagging & NER using the Spacy library**

**POS Tagging** (refers to Part-of-Speech Tagging) labels each word with its grammatical role in the sentence: noun, verb, adjective, adverb, pronouns, etc..

**NER** (Named Entity Recognition) finds and classifies real-world named things in the text, such as people, places, organizations, dates, and amounts

In [143]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [144]:
pos_tags, entities = [], []

for doc in nlp.pipe(original_text['Text'], batch_size=1000):   # one pass, processed in batches
    pos_tags.append([(tok.text, tok.pos_) for tok in doc if not tok.is_punct and not tok.is_space])
    entities.append([(ent.text, ent.label_) for ent in doc.ents])

text['POS'] = pos_tags
text['NER'] = entities

In [147]:
text[:4]

,Text,Tokens,Sentences,Tokens_no_stop,Sentences_no_stop,Tokens_stem,POS,NER
0,originally written by marta kauffman and david...,"[originally, written, by, marta, kauffman, and...",[originally written by marta kauffman and davi...,"[originally, written, marta, kauffman, david, ...",[originally written marta kauffman david crane],"[origin, written, by, marta, kauffman, and, da...","[(Originally, ADV), (written, VERB), (by, ADP)...","[(Marta Kauffman, PERSON), (David Crane, PERSON)]"
1,transcribed by guineapig,"[transcribed, by, guineapig]",[transcribed by guineapig],"[transcribed, guineapig]",[transcribed guineapig],"[transcrib, by, guineapig]","[(Transcribed, PROPN), (by, ADP), (guineapig, ...","[(Transcribed, ORG)]"
2,central perk all present except rachel and ross,"[central, perk, all, present, except, rachel, ...","[central perk, all present except rachel and r...","[central, perk, present, except, rachel, ross]","[central perk, present except rachel ross]","[central, perk, all, present, except, rachel, ...","[(CENTRAL, PROPN), (PERK, PROPN), (ALL, PRON),...","[(ROSS, ORG)]"
3,there's nothing to tell he's just some guy i ...,"[there's, nothing, to, tell, he's, just, some,...","[there's nothing to tell, he's just some guy i...","[there's, nothing, tell, guy, work]","[there's nothing tell, guy work]","[there', noth, to, tell, he', just, some, guy,...","[(There, PRON), ('s, VERB), (nothing, PRON), (...",[]


### **Questions - Rachel**

Choose the appropriate preprocessing step that are important to address each question.

#### **Question 1:** On average, how many sentences and words does your character have to speak per episode? Does this deviate across seasons?

First, we start by concatenating the original datafarame with the preprocessed `text` dataframe, enabling us to only keep the rows where **Rachel** is involved:

In [153]:
df = df.rename(columns={'Text': 'Original_Text'})
full_df = pd.concat([df, text], axis=1)
full_df[:1]

,Original_Text,Speaker,Episode,Season,Show,Text,Tokens,Sentences,Tokens_no_stop,Sentences_no_stop,Tokens_stem,POS,NER
0,Originally written by Marta Kauffman and David...,NaN,Episode-01-The One Where Monica Gets a New Roo...,Season-01,Friends,originally written by marta kauffman and david...,"[originally, written, by, marta, kauffman, and...",[originally written by marta kauffman and davi...,"[originally, written, marta, kauffman, david, ...",[originally written marta kauffman david crane],"[origin, written, by, marta, kauffman, and, da...","[(Originally, ADV), (written, VERB), (by, ADP)...","[(Marta Kauffman, PERSON), (David Crane, PERSON)]"


We filter to keep only **Rachel's** lines and store it in `rachel_df`:

In [162]:
rachel_df = full_df[full_df['Speaker'].isin(['Rachel', 'RACH', 'RACHEL'])].copy()
rachel_df[:1]

,Original_Text,Speaker,Episode,Season,Show,Text,Tokens,Sentences,Tokens_no_stop,Sentences_no_stop,Tokens_stem,POS,NER
41,Oh God Monica hi! I just went to your buildin...,RACHEL,Episode-01-The One Where Monica Gets a New Roo...,Season-01,Friends,oh god monica hi i just went to your building...,"[oh, god, monica, hi, i, just, went, to, your,...","[oh god monica hi, i just went to your buildin...","[oh, god, monica, hi, went, building, guy, big...","[oh god monica hi, went building guy big hamme...","[oh, god, monica, hi, i, just, went, to, your,...","[(Oh, INTJ), (God, PROPN), (Monica, PROPN), (h...","[(Monica, PERSON)]"


To compute how much Rachel speaks per episode, we:

1. **Count per line:** for each of Rachel's lines, the number of words is the length of its `Tokens` list and the number of sentences is the length of its `Sentences` list. We use the versions **with** stopwords, since we want to count everything she says.
2. **Sum per episode:** we add up these counts for every episode (grouped by `Season` and `Episode`), giving the total number of words and sentences Rachel speaks in that episode.
3. **Include episodes without Rachel:** episodes where she has no lines are given a count of 0, so the average is taken over *all* episodes of the show rather than only those she appears in.
4. **Average over episodes:** the mean of these per-episode totals gives the average number of words and sentences Rachel speaks per episode.


In [163]:
# Average Length 'Tokens' and Average Length 'Sentences', episodes where she does not appear must be counted as zero

# Number of words and sentences in each of Rachel's lines (stopwords kept)
rachel_df['n_words'] = rachel_df['Tokens'].str.len()
rachel_df['n_sentences'] = rachel_df['Sentences'].str.len()

# Total per episode. Reindexing on every episode in the show gives 0 to episodes without Rachel
all_episodes = pd.MultiIndex.from_frame(full_df[['Season', 'Episode']].drop_duplicates())
per_episode = (rachel_df
               .groupby(['Season', 'Episode'])[['n_words', 'n_sentences']]
               .sum()
               .reindex(all_episodes, fill_value=0))

print(f"Episodes: {len(per_episode)}")
print(f"Average words per episode:     {per_episode['n_words'].mean():.1f}")
print(f"Average sentences per episode: {per_episode['n_sentences'].mean():.1f}")

Episodes: 225
Average words per episode:     484.7
Average sentences per episode: 74.6


Performing the same task but this time distinguishing between the different **seasons** of the show:

In [165]:
per_season = per_episode.groupby(level='Season').agg(['mean'])
per_season

,n_words,n_sentences
,mean,mean
Season,,
Season-01,392.458333,67.541667
Season-02,390.869565,55.043478
Season-03,393.120000,58.960000
Season-04,477.956522,70.739130
Season-05,507.304348,80.130435
Season-06,549.913043,83.956522
Season-07,609.608696,89.521739
Season-08,591.260870,92.086957
